<img src="https://ga-curriculum.github.io/universal-resources-internal/ga-logos/ga-red-cog-72x72.png" alt="" style="float: left; margin: 20px 20px 0 0; height: 55px">

## Neural Networks for NLP


_Author:_ Tim Book (and technically also Francois Chollet, mostly)

In [ ]:
import os
import shutil
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# Get IMDB Review Data
Data are IMDB reviews (text) with sentiment as target (positive or negative). The data comes from Stanford, where they provide us with training data, testing data, and unsupervised data which we delete. We also will pull 20% of the training data into a new "validation" folder.

In [ ]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup

In [ ]:
train_dir = f'aclImdb/train'
val_dir = f'aclImdb/val'

if not os.path.exists(val_dir):
  os.mkdir(val_dir)

for cat in ['pos', 'neg']:
  train_files = os.listdir(os.path.join(train_dir, cat))
  n_train = len(train_files)

  # Separate 20% of training for validation
  np.random.seed(123)
  val_files = np.random.choice(train_files, size=int(0.2*n_train), replace=False)

  # Create val directory
  if not os.path.exists(os.path.join(val_dir, cat)):
    os.mkdir(os.path.join(val_dir, cat))

  # Move files from train to val
  for f in val_files:
    shutil.move(f'{train_dir}/{cat}/{f}', f'{val_dir}/{cat}/{f}')

# Get GloVe Word Vectors
The `wget` command pulls data from the web. We unzip this folder and read in word vectors as Python objects.

In [ ]:
# UNCOMMENT BELOW AND RUN ASAP!
# This cell can take up to 5 minutes to run, as it's downloading and unzipping
# a large file. Once run, please re-comment it so you don't accidentally run
# it twice!

!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

# Create Keras Datasets
These are special datasets that pull data from a specific folder structure which we have curated.

In [ ]:
batch_size = 32
train_ds = keras.utils.text_dataset_from_directory('aclImdb/train', batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory('aclImdb/val', batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory('aclImdb/test', batch_size=batch_size)

# Get dataset of only text (no labels)
text_only_train_ds = train_ds.map(lambda x, y: x)

In [ ]:
max_length = 600
max_tokens = 20_000

# Create an object that can process strings into integet incodings based on our
# trainind data vocabulary.
text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens, 
    output_mode='int', 
    output_sequence_length=max_length
)
text_vectorization.adapt(text_only_train_ds)

In [ ]:
text_vectorization('the cat sat on the mat')[:8]

In [ ]:
text_vectorization('the xyz sat on the mat')[:8]

In [ ]:
# Create datasets that turn our text input into integer encodings
int_mapper = lambda x, y: (text_vectorization(x), y)

int_train_ds = train_ds.map(int_mapper, num_parallel_calls=4)
int_val_ds = val_ds.map(int_mapper, num_parallel_calls=4)
int_test_ds = test_ds.map(int_mapper, num_parallel_calls=4)

In [ ]:
inputs = keras.Input(shape=(None,))
embedded = layers.Embedding(
    input_dim=max_tokens,  # Maximum of 600 tokens per sequence
    output_dim=256,        # Output of 256-length word vecs
    mask_zero=True         # This tells our model to skip over 0-padding tokens
)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer='rmsprop', loss='bce', metrics=['acc'])
model.summary()

In [ ]:
# DO NOT RUN DURING CLASS!!!
# This cell can take between 5 and 10 minutes to run - run on your own time.
# Spoiler alert: Testing accuracy of about 87%
# model.fit(
#     int_train_ds,
#     validation_data=int_val_ds,
#     epochs=10
# )

# model.evaluate(int_test_ds)

# Fitting the same model with GloVe's pretrained embeddings

In [ ]:
embeddings_ix = {}

with open('./glove.6B.100d.txt', 'r') as f:
  for line in f:
    word, coefs = line.split(maxsplit=1)
    coefs = np.fromstring(coefs, 'f', sep=' ')
    embeddings_ix[word] = coefs

In [ ]:
embeddings_ix['cat'][:4]

In [ ]:
embedding_dim = 100

vocab = text_vectorization.get_vocabulary()
word_ix = dict(enumerate(vocab))
word_ix = {word: i for i, word in word_ix.items()}

In [ ]:
# 0 and 1 occupy special places in this architecture!
print(word_ix[''])
print(word_ix['[UNK]'])
print(word_ix['cat'])

In [ ]:
# Create a huge embedding matrix based on GloVe - no need to train it ourself!
# 1 row = 1 word vector
embedding_mx = np.zeros((max_tokens, embedding_dim))

for word, i in word_ix.items():
  if i < max_tokens:
    embed_vector = embeddings_ix.get(word)
  
  if embed_vector is not None:
    embedding_mx[i, :] = embed_vector

In [ ]:
embedding_mx[1112, :5]

In [ ]:
embeddings_ix['cat'][:5]

In [ ]:
# Create a "frozen" embedding layer
glove_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_mx),
    trainable=False,
    mask_zero=True
)

In [ ]:
inputs = keras.Input(shape=(None,))
embedded = glove_layer(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer='rmsprop', loss='bce', metrics=['acc'])
model.summary()

In [ ]:
model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=10
)

In [ ]:
model.evaluate(int_test_ds)